# Projeto 3 - Iluminacao (ambiente, difusa e especular)

Adaptacao do Projeto 2. A cena combina um **ambiente externo** (terreno `outside.obj`,
skybox e a **bicicleta** que transladando carrega uma fonte de luz) com um **ambiente
interno** (uma casa oca retangular contendo uma **lanterna** pendurada no teto e um
**abajur** primitivo sobre uma mesa). Cada fonte de luz tem seu interruptor de teclado.


Nome: Pedro Louro Fernandes

NUSP: 13672446


In [1]:
import math
import os
import ctypes
import glfw
import numpy as np
from OpenGL.GL import *
from PIL import Image

# ---------------------------------------------------------------------------
# VERTEX SHADER
# Agora cada vertice traz posicao, coordenada de textura E normal. A normal e o
# vetor perpendicular a superficie, indispensavel para calcular iluminacao
# difusa/especular. Calculamos a posicao do fragmento em espaco de mundo
# (frag_pos) e transformamos a normal pela matriz normal (inversa-transposta do
# modelo) para que escalas nao-uniformes nao a distorcam.
# ---------------------------------------------------------------------------
SHADER_VS = """
#version 330 core
in vec3 posicao;
in vec2 texcoord;
in vec3 normal;

uniform mat4 modelo;
uniform mat4 visao;
uniform mat4 projecao;
uniform mat3 matriz_normal;

out vec2 v_tex;
out vec3 v_normal;
out vec3 v_frag_pos;

void main() {
    vec4 pos_mundo = modelo * vec4(posicao, 1.0);
    v_frag_pos = pos_mundo.xyz;
    v_normal = normalize(matriz_normal * normal);
    v_tex = texcoord;
    gl_Position = projecao * visao * pos_mundo;
}
"""

# ---------------------------------------------------------------------------
# FRAGMENT SHADER
# Implementa o modelo de Phong com ate 3 fontes de luz. Cada luz tem posicao,
# cor, um flag liga/desliga (intensidade) e um "ambiente" (0 = externo,
# 1 = interno). O objeto desenhado tambem informa o seu ambiente; uma luz so
# contribui se o seu ambiente bate com o do objeto. Assim a luz da bike (externa)
# nunca ilumina a casa, e a lanterna/abajur (internas) nunca iluminam o terreno.
#
# Os coeficientes de reflexao difusa (kd) e especular (ks) e o brilho
# (shininess) sao PROPRIOS de cada objeto, passados como uniforms - nao vem de
# arquivos .mtl. A luz ambiente global tambem e controlavel via teclado.
# ---------------------------------------------------------------------------
SHADER_FS = """
#version 330 core
in vec2 v_tex;
in vec3 v_normal;
in vec3 v_frag_pos;

#define MAX_LUZES 3

uniform vec4 cor;
uniform sampler2D tex;
uniform int usa_textura;

// Material proprio do objeto (NAO vem de .mtl)
uniform float kd;          // intensidade da reflexao difusa
uniform float ks;          // intensidade da reflexao especular
uniform float shininess;   // expoente especular

// Camera (para o termo especular)
uniform vec3 cam_pos;

// Luz ambiente global
uniform float luz_ambiente;

// Fontes de luz
uniform int num_luzes;
uniform vec3 luz_pos[MAX_LUZES];
uniform vec3 luz_cor[MAX_LUZES];
uniform float luz_on[MAX_LUZES];      // 1.0 ligada, 0.0 desligada
uniform int  luz_ambiente_id[MAX_LUZES]; // 0 externo, 1 interno

// Mascara de ambientes do objeto: bit0 = externo, bit1 = interno.
// Um objeto pode pertencer a ambos (ex.: a casa, externa por fora e interna por dentro).
uniform int objeto_ambiente;

// Para objetos que sao "casca" com dois lados (a casa): quando split_casa==1,
// uma luz EXTERNA so ilumina faces cuja normal aponta para FORA do centro do
// objeto, e uma luz INTERNA so ilumina faces que apontam para DENTRO. Isso evita
// a luz externa "vazar" para as faces internas das paredes (nao temos sombras).
uniform int split_casa;
uniform vec3 centro_obj;

// Objetos que sao a propria fonte de luz "brilham" (sem sombreamento)
uniform int emissivo;

out vec4 fragColor;

void main() {
    vec4 base = (usa_textura == 1) ? texture(tex, v_tex) : cor;

    // Objeto emissivo: aparece sempre com sua cor cheia (representa a lampada)
    if (emissivo == 1) {
        fragColor = base;
        return;
    }

    vec3 N = normalize(v_normal);
    vec3 V = normalize(cam_pos - v_frag_pos);

    // Componente ambiente (afeta todos os objetos por igual)
    vec3 resultado = luz_ambiente * base.rgb;

    for (int i = 0; i < num_luzes; ++i) {
        // a luz so age se o objeto pertence ao ambiente dela
        // (luz_ambiente_id 0 -> bit0 externo; 1 -> bit1 interno)
        int bit = (luz_ambiente_id[i] == 0) ? 1 : 2;
        if ((objeto_ambiente & bit) == 0) continue;

        // separacao de faces (so para a casa)
        if (split_casa == 1) {
            float lado = dot(N, v_frag_pos - centro_obj);
            // luz externa (id 0) exige face apontando para fora (lado > 0)
            // luz interna (id 1) exige face apontando para dentro (lado < 0)
            if (luz_ambiente_id[i] == 0 && lado <= 0.0) continue;
            if (luz_ambiente_id[i] == 1 && lado >= 0.0) continue;
        }
        if (luz_on[i] < 0.5) continue;

        vec3 L = luz_pos[i] - v_frag_pos;
        float dist = length(L);
        L = L / max(dist, 0.0001);

        // atenuacao suave com a distancia
        float aten = 1.0 / (1.0 + 0.0008 * dist * dist);

        // difusa (Lambert)
        float diff = max(dot(N, L), 0.0);
        vec3 difusa = kd * diff * luz_cor[i] * base.rgb;

        // especular (Phong) - so quando a face esta voltada para a luz
        vec3 especular = vec3(0.0);
        if (diff > 0.0) {
            vec3 R = reflect(-L, N);
            float spec = pow(max(dot(V, R), 0.0), max(shininess, 1.0));
            especular = ks * spec * luz_cor[i];
        }

        resultado += (difusa + especular) * aten;
    }

    fragColor = vec4(resultado, base.a);
}
"""

DEFAULT_COLOR = (0.8, 0.8, 0.8, 1.0)


## Funcoes de matriz

Base matematica de transformacoes 3D. Identicas ao Projeto 2, com o acrescimo
de `mat3` e `normal_matrix`: a matriz normal e a inversa-transposta do bloco 3x3
do modelo, usada no shader para transformar normais corretamente sob escala.


In [ ]:
def ident():
    return np.eye(4, dtype=np.float32)

def transl(tx, ty, tz):
    m = ident()
    m[0, 3], m[1, 3], m[2, 3] = tx, ty, tz
    return m

def esc(sx, sy, sz):
    m = ident()
    m[0, 0], m[1, 1], m[2, 2] = sx, sy, sz
    return m

def rot_x(a):
    c, s = math.cos(a), math.sin(a)
    m = ident()
    m[1, 1], m[1, 2], m[2, 1], m[2, 2] = c, -s, s, c
    return m

def rot_y(a):
    c, s = math.cos(a), math.sin(a)
    m = ident()
    m[0, 0], m[0, 2], m[2, 0], m[2, 2] = c, s, -s, c
    return m

def rot_z(a):
    c, s = math.cos(a), math.sin(a)
    m = ident()
    m[0, 0], m[0, 1], m[1, 0], m[1, 1] = c, -s, s, c
    return m

def perspectiva(fovy, asp, near, far):
    f = 1.0 / math.tan(fovy / 2.0)
    m = np.zeros((4, 4), dtype=np.float32)
    m[0, 0], m[1, 1] = f / asp, f
    m[2, 2], m[2, 3], m[3, 2] = (far + near) / (near - far), (2.0 * far * near) / (near - far), -1.0
    return m

def look_at(eye, target, up=(0.0, 1.0, 0.0)):
    f = target - eye
    f = f / np.linalg.norm(f)
    u = np.array(up, dtype=np.float32)
    u = u / np.linalg.norm(u)
    s = np.cross(f, u)
    s = s / np.linalg.norm(s)
    u = np.cross(s, f)

    m = ident()
    m[0, 0], m[0, 1], m[0, 2] = s[0], s[1], s[2]
    m[1, 0], m[1, 1], m[1, 2] = u[0], u[1], u[2]
    m[2, 0], m[2, 1], m[2, 2] = -f[0], -f[1], -f[2]
    return m @ transl(-eye[0], -eye[1], -eye[2])

def normal_matrix(modelo):
    sup = np.array(modelo[:3, :3], dtype=np.float32)
    try:
        inv_t = np.linalg.inv(sup).T
    except np.linalg.LinAlgError:
        inv_t = sup
    return np.array(inv_t, dtype=np.float32)


## Carregamento de assets e skybox

- `ler_mtl`: le `.mtl` apenas para extrair o mapa de textura (`map_Kd`) e a cor difusa
  base usada como **cor de superficie** (albedo). Os coeficientes de iluminacao
  (difuso/especular) NAO sao lidos do `.mtl` - sao definidos por objeto no codigo;
- `carregar_textura`: envia imagem para a GPU como textura 2D com mipmaps;
- `preparar_materiais`: resolve texturas/cores base de cada material;
- `carregar_obj`: parser de `.obj` com vertices, UVs, **normais** (`vn`) e faces
  trianguladas. Quando o `.obj` nao traz normais, elas sao calculadas por face
  (produto vetorial das arestas) para que a iluminacao funcione mesmo assim.
  Retorna vertices interleaved (x,y,z,u,v,nx,ny,nz);
- `skybox_vertices`: esfera equiretangular para o ceu.


In [3]:
def ler_mtl(caminho):
    materiais = {}
    atual = None
    if not os.path.exists(caminho):
        return materiais

    with open(caminho, "r", encoding="utf-8", errors="ignore") as arq:
        for linha in arq:
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue
            chave, *resto = linha.split(maxsplit=1)
            valor = resto[0] if resto else ""
            if chave == "newmtl":
                atual = valor
                materiais[atual] = {}
            elif atual is not None and chave == "Kd":
                partes = [float(x) for x in valor.split()[:3]]
                materiais[atual]["Kd"] = (partes[0], partes[1], partes[2])
            elif atual is not None and chave == "map_Kd":
                materiais[atual]["map_Kd"] = valor
    return materiais

def carregar_textura(caminho, clamp=False):
    if not os.path.exists(caminho):
        return 0
    img = Image.open(caminho).convert("RGBA")
    if not clamp:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    dados = img.tobytes()
    wrap = GL_CLAMP_TO_EDGE if clamp else GL_REPEAT
    tex_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_2D, tex_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, wrap)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, wrap)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, img.width, img.height, 0, GL_RGBA, GL_UNSIGNED_BYTE, dados)
    glGenerateMipmap(GL_TEXTURE_2D)
    glBindTexture(GL_TEXTURE_2D, 0)
    return tex_id

def preparar_materiais(materiais, base_dir):
    saida = {}
    for nome, mat in materiais.items():
        cor = mat.get("Kd", (1.0, 1.0, 1.0))
        tex = 0
        mapa = mat.get("map_Kd")
        if not mapa and os.path.basename(base_dir).lower() == "lanterna":
            fallback_tex = "lantern_Base_Color.jpg"
            if os.path.exists(os.path.join(base_dir, fallback_tex)):
                mapa = fallback_tex
        if mapa:
            tex = carregar_textura(os.path.join(base_dir, mapa))
        saida[nome] = {"cor": (cor[0], cor[1], cor[2], 1.0), "tex": tex}
    return saida

def carregar_obj(caminho):
    """Parser de .obj. Saida interleaved: x,y,z, u,v, nx,ny,nz (8 floats/vertice)."""
    base_dir = os.path.dirname(caminho)
    materiais = {}
    vertices = []
    uvs = []
    normais = []
    dados = {}
    atual = "__default__"
    dados[atual] = []

    def parse_indice(valor, tamanho):
        idx = int(valor)
        return idx - 1 if idx > 0 else tamanho + idx

    with open(caminho, "r", encoding="utf-8", errors="ignore") as arq:
        for linha in arq:
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue
            partes = linha.split()
            cmd = partes[0]

            if cmd == "v":
                vertices.append((float(partes[1]), float(partes[2]), float(partes[3])))
            elif cmd == "vt":
                uvs.append((float(partes[1]), float(partes[2])))
            elif cmd == "vn":
                normais.append((float(partes[1]), float(partes[2]), float(partes[3])))
            elif cmd == "mtllib":
                mtl_path = os.path.join(base_dir, " ".join(partes[1:]))
                materiais.update(ler_mtl(mtl_path))
            elif cmd == "usemtl":
                atual = " ".join(partes[1:])
                dados.setdefault(atual, [])
            elif cmd == "f":
                indices = []
                for tok in partes[1:]:
                    campos = tok.split("/")
                    vi = parse_indice(campos[0], len(vertices))
                    vti = None
                    ni = None
                    if len(campos) > 1 and campos[1]:
                        vti = parse_indice(campos[1], len(uvs))
                    if len(campos) > 2 and campos[2]:
                        ni = parse_indice(campos[2], len(normais))
                    indices.append((vi, vti, ni))

                # triangulacao em leque
                for i in range(1, len(indices) - 1):
                    tri = (indices[0], indices[i], indices[i + 1])

                    # se faltarem normais no arquivo, calcula a normal da face
                    p0 = np.array(vertices[tri[0][0]], dtype=np.float32)
                    p1 = np.array(vertices[tri[1][0]], dtype=np.float32)
                    p2 = np.array(vertices[tri[2][0]], dtype=np.float32)
                    face_n = np.cross(p1 - p0, p2 - p0)
                    norma = np.linalg.norm(face_n)
                    face_n = face_n / norma if norma > 1e-8 else np.array([0.0, 1.0, 0.0], dtype=np.float32)

                    for vi, vti, ni in tri:
                        px, py, pz = vertices[vi]
                        if vti is not None and vti < len(uvs):
                            tu, tv = uvs[vti]
                        else:
                            tu, tv = 0.0, 0.0
                        if ni is not None and ni < len(normais):
                            nx, ny, nz = normais[ni]
                        else:
                            nx, ny, nz = face_n
                        dados[atual].extend([px, py, pz, tu, tv, nx, ny, nz])

    if vertices:
        arr = np.array(vertices, dtype=np.float32)
        min_v = np.min(arr, axis=0)
        max_v = np.max(arr, axis=0)
    else:
        min_v = np.array([0.0, 0.0, 0.0], dtype=np.float32)
        max_v = np.array([0.0, 0.0, 0.0], dtype=np.float32)

    todos = []
    batches = []
    inicio = 0
    for mat, arr in dados.items():
        if not arr:
            continue
        todos.extend(arr)
        count = len(arr) // 8
        batches.append({"material": mat, "start": inicio, "count": count})
        inicio += count

    return np.array(todos, dtype=np.float32), batches, materiais, (min_v, max_v)

def skybox_vertices():
    """Esfera equiretangular. Inclui normais (apontando para dentro nao importa:
    a skybox e desenhada como emissiva, sem iluminacao)."""
    def pt(phi, th):
        return (math.sin(phi)*math.cos(th), math.cos(phi), math.sin(phi)*math.sin(th))

    stacks, slices = 16, 32
    out = []
    for i in range(stacks):
        phi0, phi1 = math.pi * i / stacks, math.pi * (i+1) / stacks
        for j in range(slices):
            th0, th1 = 2*math.pi * j / slices, 2*math.pi * (j+1) / slices
            u0, u1 = j/slices, (j+1)/slices
            v0, v1 = i/stacks, (i+1)/stacks
            p00, p10 = pt(phi0, th0), pt(phi1, th0)
            p01, p11 = pt(phi0, th1), pt(phi1, th1)
            # normal qualquer (skybox e emissiva)
            n = (0.0, 1.0, 0.0)
            out.extend([*p00, u0, v0, *n, *p10, u0, v1, *n, *p11, u1, v1, *n])
            out.extend([*p00, u0, v0, *n, *p11, u1, v1, *n, *p01, u1, v0, *n])
    return np.array(out, dtype=np.float32)


## Geometria primitiva (casa, abajur e marcadores de luz)

Como a casa e o abajur nao vem de arquivos, geramos sua geometria com formas
primitivas. Todas as funcoes produzem o mesmo formato interleaved (8 floats por
vertice) usado pelo parser de `.obj`, ja com normais corretas.

- `cubo`: caixa unitaria centrada na origem, com normais por face. Serve de bloco
  de construcao para as paredes e o piso/teto da casa, alem dos pequenos cubos
  que marcam visualmente as fontes de luz;
- `casinha`: monta uma casa retangular oca (4 paredes + piso + teto) deixando um
  vao de porta numa das paredes (a parede frontal e dividida em segmentos ao redor
  da abertura). Um teto simples de duas aguas e adicionado por cima;
- `cilindro` e `cone`: usados para montar o abajur (base + haste + cupula conica);
- `esfera_simples`: pequena esfera para a "lampada" do abajur.

Cada primitiva ja devolve `(verts, batches)` com um unico material nomeado, para
encaixar direto no pipeline de desenho.


In [ ]:
def _quad(p0, p1, p2, p3, n):
    uv = [(0,0),(1,0),(1,1),(0,1)]
    pts = [p0, p1, p2, p3]
    out = []
    for i in (0, 1, 2, 0, 2, 3):
        x, y, z = pts[i]
        u, v = uv[i]
        out.extend([x, y, z, u, v, n[0], n[1], n[2]])
    return out

def cubo(cx=0, cy=0, cz=0, sx=1, sy=1, sz=1):
    hx, hy, hz = sx/2, sy/2, sz/2
    x0, x1 = cx-hx, cx+hx
    y0, y1 = cy-hy, cy+hy
    z0, z1 = cz-hz, cz+hz
    out = []
    out += _quad((x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1), (0,0,1))
    out += _quad((x1,y0,z0),(x0,y0,z0),(x0,y1,z0),(x1,y1,z0), (0,0,-1))
    # direita (+x) / esquerda (-x)
    out += _quad((x1,y0,z1),(x1,y0,z0),(x1,y1,z0),(x1,y1,z1), (1,0,0))
    out += _quad((x0,y0,z0),(x0,y0,z1),(x0,y1,z1),(x0,y1,z0), (-1,0,0))
    # topo (+y) / base (-y)
    out += _quad((x0,y1,z1),(x1,y1,z1),(x1,y1,z0),(x0,y1,z0), (0,1,0))
    out += _quad((x0,y0,z0),(x1,y0,z0),(x1,y0,z1),(x0,y0,z1), (0,-1,0))
    return np.array(out, dtype=np.float32)

def primitiva(verts, material):
    count = len(verts) // 8
    return verts, [{"material": material, "start": 0, "count": count}]

def casinha(largura=10.0, profundidade=10.0, altura=5.0,
             espessura=0.3, porta_larg=2.2, porta_alt=3.2):
    out = []
    hw, hd = largura/2, profundidade/2
    e = espessura

    # piso e teto plano (lajes finas)
    out += list(cubo(0, 0, 0, largura, e, profundidade))
    out += list(cubo(0, altura, 0, largura, e, profundidade))

    # parede de tras (-z) e laterais (+/-x)
    out += list(cubo(0, altura/2, -hd, largura, altura, e))
    out += list(cubo(hw, altura/2, 0, e, altura, profundidade))
    out += list(cubo(-hw, altura/2, 0, e, altura, profundidade))

    # parede frontal (+z) com vao de porta: 2 colunas + verga
    col = (largura - porta_larg) / 2.0
    # coluna esquerda
    out += list(cubo(-(porta_larg/2 + col/2), altura/2, hd, col, altura, e))
    # coluna direita
    out += list(cubo((porta_larg/2 + col/2), altura/2, hd, col, altura, e))
    # verga acima da porta
    verga_h = altura - porta_alt
    out += list(cubo(0, porta_alt + verga_h/2, hd, porta_larg, verga_h, e))

    verts = np.array(out, dtype=np.float32)
    v, b = primitiva(verts, "casa")

    # telhado de duas aguas (dois quads inclinados + duas empenas triangulares)
    topo = altura + e/2
    cumeeira = topo + 2.5
    beiral = hw + 0.6
    tout = []
    # agua esquerda e direita
    tout += _quad((-beiral, topo, hd+0.6), (-beiral, topo, -hd-0.6),
                  (0, cumeeira, -hd-0.6), (0, cumeeira, hd+0.6), (-0.7, 0.7, 0))
    tout += _quad((0, cumeeira, hd+0.6), (0, cumeeira, -hd-0.6),
                  (beiral, topo, -hd-0.6), (beiral, topo, hd+0.6), (0.7, 0.7, 0))
    # empenas (triangulos) frente e tras
    def tri(a, bb, c, n):
        o = []
        for x, y, z in (a, bb, c):
            o.extend([x, y, z, 0, 0, n[0], n[1], n[2]])
        return o
    tout += tri((-beiral, topo, hd+0.6), (beiral, topo, hd+0.6), (0, cumeeira, hd+0.6), (0,0,1))
    tout += tri((beiral, topo, -hd-0.6), (-beiral, topo, -hd-0.6), (0, cumeeira, -hd-0.6), (0,0,-1))
    tverts = np.array(tout, dtype=np.float32)

    # junta casa + telhado num unico buffer com dois batches
    todos = np.concatenate([verts, tverts])
    n_casa = len(verts) // 8
    n_tel = len(tverts) // 8
    batches = [
        {"material": "casa", "start": 0, "count": n_casa},
        {"material": "telhado", "start": n_casa, "count": n_tel},
    ]
    return todos, batches

def cilindro(raio=0.5, altura=1.0, cy=0.0, seg=24, material="abajur"):
    out = []
    y0, y1 = cy, cy + altura
    for i in range(seg):
        a0 = 2*math.pi*i/seg
        a1 = 2*math.pi*(i+1)/seg
        x0, z0 = raio*math.cos(a0), raio*math.sin(a0)
        x1, z1 = raio*math.cos(a1), raio*math.sin(a1)
        # lateral
        n0 = (math.cos(a0), 0, math.sin(a0))
        n1 = (math.cos(a1), 0, math.sin(a1))
        out += [x0,y0,z0, 0,0, *n0,  x1,y0,z1, 0,0, *n1,  x1,y1,z1, 0,0, *n1]
        out += [x0,y0,z0, 0,0, *n0,  x1,y1,z1, 0,0, *n1,  x0,y1,z0, 0,0, *n0]
        # tampa de cima e de baixo
        out += [0,y1,0, 0,0, 0,1,0,  x0,y1,z0, 0,0, 0,1,0,  x1,y1,z1, 0,0, 0,1,0]
        out += [0,y0,0, 0,0, 0,-1,0, x1,y0,z1, 0,0, 0,-1,0, x0,y0,z0, 0,0, 0,-1,0]
    return primitiva(np.array(out, dtype=np.float32), material)

def cone(raio=1.0, altura=1.5, cy=0.0, seg=24, material="cupula"):
    out = []
    apex = (0, cy+altura, 0)
    for i in range(seg):
        a0 = 2*math.pi*i/seg
        a1 = 2*math.pi*(i+1)/seg
        x0, z0 = raio*math.cos(a0), raio*math.sin(a0)
        x1, z1 = raio*math.cos(a1), raio*math.sin(a1)
        # normal aproximada apontando para fora/cima
        n0 = (math.cos(a0), 0.4, math.sin(a0))
        n1 = (math.cos(a1), 0.4, math.sin(a1))
        out += [x0,cy,z0, 0,0, *n0,  x1,cy,z1, 0,0, *n1,  apex[0],apex[1],apex[2], 0,0, 0,1,0]
    return primitiva(np.array(out, dtype=np.float32), material)

def esfera_simples(raio=0.3, seg=16, material="bulbo"):
    out = []
    for i in range(seg):
        phi0, phi1 = math.pi*i/seg, math.pi*(i+1)/seg
        for j in range(seg):
            th0, th1 = 2*math.pi*j/seg, 2*math.pi*(j+1)/seg
            def p(phi, th):
                return (raio*math.sin(phi)*math.cos(th),
                        raio*math.cos(phi),
                        raio*math.sin(phi)*math.sin(th))
            def n(phi, th):
                return (math.sin(phi)*math.cos(th), math.cos(phi), math.sin(phi)*math.sin(th))
            a, b2, c, d = p(phi0,th0), p(phi1,th0), p(phi1,th1), p(phi0,th1)
            na, nb, nc, nd = n(phi0,th0), n(phi1,th0), n(phi1,th1), n(phi0,th1)
            out += [*a,0,0,*na, *b2,0,0,*nb, *c,0,0,*nc]
            out += [*a,0,0,*na, *c,0,0,*nc, *d,0,0,*nd]
    return primitiva(np.array(out, dtype=np.float32), material)


def criar_mesa(tampo_larg=3.0, tampo_prof=2.0, tampo_esp=0.18,
               altura=2.6, perna=0.22):
    out = []
    y_tampo = altura - tampo_esp / 2.0
    out += list(cubo(0, y_tampo, 0, tampo_larg, tampo_esp, tampo_prof))
    hx = tampo_larg/2 - perna/2 - 0.1
    hz = tampo_prof/2 - perna/2 - 0.1
    h_perna = altura - tampo_esp
    for sx in (-1, 1):
        for sz in (-1, 1):
            out += list(cubo(sx*hx, h_perna/2, sz*hz, perna, h_perna, perna))
    return primitiva(np.array(out, dtype=np.float32), "mesa")

def criar_cadeira(largura=1.2, prof=1.2, altura_assento=1.3, altura_encosto=1.3, esp=0.16):
    out = []
    # assento
    y_ass = altura_assento - esp/2
    out += list(cubo(0, y_ass, 0, largura, esp, prof))
    # encosto (na parte de tras, -z)
    out += list(cubo(0, altura_assento + altura_encosto/2, -prof/2 + esp/2,
                     largura, altura_encosto, esp))
    # pernas
    hx = largura/2 - esp/2 - 0.05
    hz = prof/2 - esp/2 - 0.05
    for sx in (-1, 1):
        for sz in (-1, 1):
            out += list(cubo(sx*hx, altura_assento/2, sz*hz, esp, altura_assento, esp))
    return primitiva(np.array(out, dtype=np.float32), "cadeira")


## Montagem da cena e funcoes de GPU

- `montar_cena`: orquestra `carregar_obj` + `preparar_materiais` (fallback de cor);
- `montar_primitiva`: empacota geometria gerada em codigo num objeto com materiais
  de cor solida (sem textura);
- `criar_vao_vbo`: agora com stride de **8 floats** e o atributo extra `normal`;
- `desenhar_objeto`: alem da cor/textura, define por objeto os uniforms de
  iluminacao proprios (`kd`, `ks`, `shininess`), o ambiente do objeto e o flag
  emissivo, e envia a matriz normal correspondente ao modelo.


In [5]:
def montar_cena(caminho_obj):
    verts, batches, mats, bounds = carregar_obj(caminho_obj)
    base_dir = os.path.dirname(caminho_obj)
    materiais = preparar_materiais(mats, base_dir)
    for batch in batches:
        if batch["material"] not in materiais:
            materiais[batch["material"]] = {"cor": DEFAULT_COLOR, "tex": 0}
    return verts, batches, materiais, bounds

def montar_primitiva(verts, batches, cores):
    """cores: dict material -> (r,g,b,a). Cria materiais de cor solida."""
    materiais = {}
    for b in batches:
        m = b["material"]
        materiais[m] = {"cor": cores.get(m, DEFAULT_COLOR), "tex": 0}
    return verts, batches, materiais

def criar_vao_vbo(verts, loc_pos, loc_uv, loc_norm):
    vao = glGenVertexArrays(1)
    glBindVertexArray(vao)
    vbo = glGenBuffers(1)
    glBindBuffer(GL_ARRAY_BUFFER, vbo)
    glBufferData(GL_ARRAY_BUFFER, verts.nbytes, verts, GL_STATIC_DRAW)

    stride = 8 * verts.itemsize
    glEnableVertexAttribArray(loc_pos)
    glVertexAttribPointer(loc_pos, 3, GL_FLOAT, False, stride, ctypes.c_void_p(0))
    glEnableVertexAttribArray(loc_uv)
    glVertexAttribPointer(loc_uv, 2, GL_FLOAT, False, stride, ctypes.c_void_p(3 * verts.itemsize))
    glEnableVertexAttribArray(loc_norm)
    glVertexAttribPointer(loc_norm, 3, GL_FLOAT, False, stride, ctypes.c_void_p(5 * verts.itemsize))

    glBindVertexArray(0)
    return vao, vbo

def desenhar_objeto(objeto, locs, modelo, ambiente, kd, ks, shininess,
                    emissivo=0, split_casa=0, centro_obj=(0.0, 0.0, 0.0)):
    """Desenha um objeto aplicando seus proprios parametros de iluminacao.
    split_casa=1 ativa a separacao de faces interna/externa (usado na casa),
    com centro_obj sendo o centro do objeto em espaco de mundo."""
    glUniformMatrix4fv(locs["model"], 1, GL_TRUE, modelo)
    glUniformMatrix3fv(locs["nmat"], 1, GL_TRUE, normal_matrix(modelo))
    glUniform1i(locs["obj_amb"], ambiente)
    glUniform1f(locs["kd"], kd)
    glUniform1f(locs["ks"], ks)
    glUniform1f(locs["shin"], shininess)
    glUniform1i(locs["emis"], emissivo)
    glUniform1i(locs["split"], split_casa)
    glUniform3f(locs["centro"], centro_obj[0], centro_obj[1], centro_obj[2])

    glBindVertexArray(objeto["vao"])
    glActiveTexture(GL_TEXTURE0)
    for batch in objeto["batches"]:
        mat = objeto["materiais"].get(batch["material"], {"cor": DEFAULT_COLOR, "tex": 0})
        if mat["tex"] != 0:
            glUniform1i(locs["usa"], 1)
            glBindTexture(GL_TEXTURE_2D, mat["tex"])
        else:
            glUniform1i(locs["usa"], 0)
            glBindTexture(GL_TEXTURE_2D, 0)
        c = mat["cor"]
        glUniform4f(locs["cor"], c[0], c[1], c[2], c[3])
        glDrawArrays(GL_TRIANGLES, batch["start"], batch["count"])


## Entrada de teclado e atualizacao de estado

O Projeto 3 troca os controles do Projeto 2 (rotacao/translacao/escala/malha de
objeto) pelos **interruptores e ajustes de iluminacao**. Camera (WASD + setas)
permanece para navegar pela cena.

`callback_tecla` trata os eventos pontuais (PRESS):

| Tecla | Acao |
|-------|------|
| `1` | Liga/desliga a luz **externa** da bicicleta (farol) |
| `2` | Liga/desliga a **lanterna** do teto (interna) |
| `3` | Liga/desliga o **abajur** da mesa (interno) |
| `0` | Liga/desliga a translacao da bicicleta |

`aplicar_controles` (continuo, por frame) trata camera e os ajustes graduais:

| Tecla | Acao |
|-------|------|
| `K` / `L` | Diminui / aumenta a **luz ambiente** global |
| `N` / `M` | Diminui / aumenta a **reflexao difusa** (kd) de todos os objetos |
| `,` / `.` | Diminui / aumenta a **reflexao especular** (ks) de todos os objetos |

Os ajustes de kd/ks sao aplicados como multiplicador global sobre os valores
proprios de cada objeto, preservando o requisito de parametros por objeto.


In [6]:
def callback_tecla(janela, key, _scancode, action, _mods):
    estado = glfw.get_window_user_pointer(janela)
    if action == glfw.PRESS:
        estado["teclas"].add(key)
        # interruptores independentes de cada fonte de luz
        if key == glfw.KEY_1:
            estado["luz_bike_on"] = not estado["luz_bike_on"]
        elif key == glfw.KEY_2:
            estado["luz_lanterna_on"] = not estado["luz_lanterna_on"]
        elif key == glfw.KEY_3:
            estado["luz_abajur_on"] = not estado["luz_abajur_on"]
        elif key == glfw.KEY_0:
            estado["bike_move"] = not estado["bike_move"]
    elif action == glfw.RELEASE:
        estado["teclas"].discard(key)

def aplicar_controles(estado, dt):
    teclas = estado["teclas"]
    vel_mov = 18.0 * dt
    vel_rot = 1.6 * dt

    yaw = estado["cam_yaw"]
    forward = np.array([math.cos(yaw), 0.0, math.sin(yaw)], dtype=np.float32)
    right = np.array([-math.sin(yaw), 0.0, math.cos(yaw)], dtype=np.float32)

    mov = np.zeros(3, dtype=np.float32)
    mov += forward * ((glfw.KEY_W in teclas) - (glfw.KEY_S in teclas))
    mov += right * ((glfw.KEY_D in teclas) - (glfw.KEY_A in teclas))
    mov[1] += ((glfw.KEY_E in teclas) - (glfw.KEY_Q in teclas))
    estado["cam_pos"] += mov * vel_mov

    estado["cam_yaw"] += ((glfw.KEY_RIGHT in teclas) - (glfw.KEY_LEFT in teclas)) * vel_rot
    estado["cam_pitch"] += ((glfw.KEY_UP in teclas) - (glfw.KEY_DOWN in teclas)) * vel_rot
    estado["cam_pitch"] = float(np.clip(estado["cam_pitch"], -1.2, 1.2))

    # ---- ajustes de iluminacao --------------------------------------------
    # luz ambiente global (K diminui, L aumenta)
    amb_delta = ((glfw.KEY_L in teclas) - (glfw.KEY_K in teclas)) * dt * 0.4
    if amb_delta:
        estado["luz_ambiente"] = float(np.clip(estado["luz_ambiente"] + amb_delta, 0.0, 1.0))

    # reflexao difusa global (N diminui, M aumenta)
    kd_delta = ((glfw.KEY_M in teclas) - (glfw.KEY_N in teclas)) * dt * 0.6
    if kd_delta:
        estado["kd_mult"] = float(np.clip(estado["kd_mult"] + kd_delta, 0.0, 3.0))
        print(f"reflexao difusa (kd_mult) = {estado['kd_mult']:.2f}")

    # reflexao especular global (, diminui  . aumenta)
    ks_delta = ((glfw.KEY_PERIOD in teclas) - (glfw.KEY_COMMA in teclas)) * dt * 1.2
    if ks_delta:
        estado["ks_mult"] = float(np.clip(estado["ks_mult"] + ks_delta, 0.0, 5.0))
        print(f"reflexao especular (ks_mult) = {estado['ks_mult']:.2f}")

    # translacao da bike
    if estado["bike_move"]:
        estado["bike_t"] += estado["bike_dir"] * estado["bike_speed"] * dt
        if estado["bike_t"] > 1.0:
            estado["bike_t"] = 1.0
            estado["bike_dir"] = -1.0
        elif estado["bike_t"] < 0.0:
            estado["bike_t"] = 0.0
            estado["bike_dir"] = 1.0

    # limites de camera (bounding box do terreno externo, com folga vertical)
    min_v, max_v = estado["bounds"]
    margem = 0.5
    estado["cam_pos"][0] = float(np.clip(estado["cam_pos"][0], min_v[0] - margem, max_v[0] + margem))
    estado["cam_pos"][2] = float(np.clip(estado["cam_pos"][2], min_v[2] - margem, max_v[2] + margem))
    estado["cam_pos"][1] = float(np.clip(estado["cam_pos"][1], min_v[1] + 0.5, max_v[1] + 500.0))


## Funcao principal e render loop

1. **Setup**: janela GLFW, compila shaders (agora com iluminacao Phong), carrega
   `outside.obj`, a bike, a lanterna, a mesa e a cadeira do disco, e gera por codigo
   a casa oca e o abajur;
2. **Layout dos ambientes**:
   - *Externo*: terreno + skybox + bicicleta. A bike carrega um pequeno cubo
     emissivo (farol) e uma fonte de luz **externa** que a acompanha;
   - *Interno*: a casa oca posicionada na cena; dentro dela a lanterna pendurada
     no teto (fonte interna, cor quente) e o abajur sobre a mesa (fonte interna,
     cor azulada). Mesa e cadeira completam o cenario;
3. **Iluminacao**: a cada frame os uniforms de luz sao atualizados. Cada luz
   informa ambiente (0 externo / 1 interno), e o shader so a aplica a objetos do
   mesmo ambiente - cumprindo a separacao exigida no enunciado;
4. **Cleanup**: libera VAOs/VBOs/texturas/programa ao fechar.

Observacao sobre caminhos: a mesa e esperada em `mesa/table.obj` e a cadeira em
`cadeira/modern chair 11 obj.obj`, na raiz do projeto (ajuste os nomes se diferirem).
Se algum desses arquivos faltar, a cena ainda roda - apenas o objeto ausente e
omitido.


In [ ]:
def criar_programa(cod_vs, cod_fs):
    def compilar(codigo, tipo):
        shader = glCreateShader(tipo)
        glShaderSource(shader, codigo)
        glCompileShader(shader)
        if not glGetShaderiv(shader, GL_COMPILE_STATUS):
            raise RuntimeError(glGetShaderInfoLog(shader).decode("utf-8"))
        return shader
    vs = compilar(cod_vs, GL_VERTEX_SHADER)
    fs = compilar(cod_fs, GL_FRAGMENT_SHADER)
    prog = glCreateProgram()
    glAttachShader(prog, vs)
    glAttachShader(prog, fs)
    glLinkProgram(prog)
    if not glGetProgramiv(prog, GL_LINK_STATUS):
        raise RuntimeError(glGetProgramInfoLog(prog).decode("utf-8"))
    glDeleteShader(vs)
    glDeleteShader(fs)
    return prog

def carregar_opcional(caminho, loc_pos, loc_uv, loc_norm, cores=None):
    """Carrega um .obj se existir; senao retorna None. cores opcional sobrescreve
    materiais por cor solida (usado quando nao queremos texturas)."""
    if not os.path.exists(caminho):
        print(f"[aviso] arquivo nao encontrado, ignorando: {caminho}")
        return None
    verts, batches, materiais, _ = montar_cena(caminho)
    if verts.size == 0:
        print(f"[aviso] nenhum vertice em: {caminho}")
        return None
    if cores:
        for b in batches:
            materiais[b["material"]] = {"cor": cores, "tex": 0}
    vao, vbo = criar_vao_vbo(verts, loc_pos, loc_uv, loc_norm)
    return {"vao": vao, "vbo": vbo, "batches": batches, "materiais": materiais}

def main():
    if not glfw.init():
        raise RuntimeError("Erro ao inicializar GLFW")

    janela = glfw.create_window(1280, 720, "Projeto3 - Iluminacao", None, None)
    if not janela:
        glfw.terminate()
        raise RuntimeError("Erro ao criar janela")
    glfw.make_context_current(janela)

    estado = {
        "teclas": set(),
        "cam_pos": np.zeros(3, dtype=np.float32),
        "cam_yaw": 0.0,
        "cam_pitch": 0.0,
        "bounds": (np.array([-1.0, -1.0, -1.0], dtype=np.float32),
                   np.array([1.0, 1.0, 1.0], dtype=np.float32)),
        # iluminacao
        "luz_ambiente": 0.20,
        "kd_mult": 1.0,
        "ks_mult": 1.0,
        "luz_bike_on": True,
        "luz_lanterna_on": True,
        "luz_abajur_on": True,
        # bike
        "bike_t": 0.0,
        "bike_dir": 1.0,
        "bike_speed": 0.12,
        "bike_move": True,
        "bike_scale": 0.1,
        "bike_tilt": -math.pi / 2.0,
        "bike_yaw_offset": -math.pi / 2.0,
        "bike_start_pos": np.array([50, 0.65, 100], dtype=np.float32),
        "bike_end_pos": np.array([15, 0.65, 100], dtype=np.float32),
        # lanterna interna (pendurada no teto da casa)
        "lanterna_scale": 0.2,
        "lanterna_base_scale": 0.05,
        # casa
        "casa_pos": np.array([10.0, 0.0, 80.0], dtype=np.float32),
        "casa_larg": 16.0,
        "casa_prof": 14.0,
        "casa_alt": 7.0,
    }
    glfw.set_window_user_pointer(janela, estado)
    glfw.set_key_callback(janela, callback_tecla)

    prog = criar_programa(SHADER_VS, SHADER_FS)
    glUseProgram(prog)

    locs = {
        "model": glGetUniformLocation(prog, "modelo"),
        "view":  glGetUniformLocation(prog, "visao"),
        "proj":  glGetUniformLocation(prog, "projecao"),
        "nmat":  glGetUniformLocation(prog, "matriz_normal"),
        "cor":   glGetUniformLocation(prog, "cor"),
        "usa":   glGetUniformLocation(prog, "usa_textura"),
        "kd":    glGetUniformLocation(prog, "kd"),
        "ks":    glGetUniformLocation(prog, "ks"),
        "shin":  glGetUniformLocation(prog, "shininess"),
        "cam":   glGetUniformLocation(prog, "cam_pos"),
        "amb":   glGetUniformLocation(prog, "luz_ambiente"),
        "nluz":  glGetUniformLocation(prog, "num_luzes"),
        "obj_amb": glGetUniformLocation(prog, "objeto_ambiente"),
        "emis":  glGetUniformLocation(prog, "emissivo"),
        "split": glGetUniformLocation(prog, "split_casa"),
        "centro": glGetUniformLocation(prog, "centro_obj"),
    }
    loc_luz_pos = glGetUniformLocation(prog, "luz_pos")
    loc_luz_cor = glGetUniformLocation(prog, "luz_cor")
    loc_luz_on  = glGetUniformLocation(prog, "luz_on")
    loc_luz_amb = glGetUniformLocation(prog, "luz_ambiente_id")

    loc_pos = glGetAttribLocation(prog, "posicao")
    loc_uv = glGetAttribLocation(prog, "texcoord")
    loc_norm = glGetAttribLocation(prog, "normal")

    base_dir = os.getcwd()
    outside_path = os.path.join(base_dir, "outside.obj")
    lanterna_path = os.path.join(base_dir, "lanterna", "lantern_obj.obj")
    bike_path = os.path.join(base_dir, "bike", "11717_bicycle_v2_L1.obj")
    skybox_tex_path = os.path.join(base_dir, "Simple_Sky-Blue_04-512x512.png")

    if not os.path.exists(outside_path):
        raise RuntimeError("Arquivo outside.obj nao encontrado no diretorio atual")

    # --- ambiente externo: terreno -----------------------------------------
    outside_verts, outside_batches, outside_materiais, bounds = montar_cena(outside_path)
    if outside_verts.size == 0:
        raise RuntimeError("Nenhum vertice carregado de outside.obj")
    outside_vao, outside_vbo = criar_vao_vbo(outside_verts, loc_pos, loc_uv, loc_norm)
    outside = {"vao": outside_vao, "vbo": outside_vbo, "batches": outside_batches, "materiais": outside_materiais}

    # bike (externo, carrega a luz)
    bike = carregar_opcional(bike_path, loc_pos, loc_uv, loc_norm)
    # lanterna (interna, no teto)
    lanterna = carregar_opcional(lanterna_path, loc_pos, loc_uv, loc_norm)

    # --- geometria gerada em codigo ----------------------------------------
    # casa oca
    casa_v, casa_b = casinha(estado["casa_larg"], estado["casa_prof"], estado["casa_alt"])
    casa_v, casa_b, casa_m = montar_primitiva(casa_v, casa_b,
        {"casa": (0.85, 0.80, 0.72, 1.0), "telhado": (0.55, 0.27, 0.18, 1.0)})
    casa_vao, casa_vbo = criar_vao_vbo(casa_v, loc_pos, loc_uv, loc_norm)
    casa = {"vao": casa_vao, "vbo": casa_vbo, "batches": casa_b, "materiais": casa_m}

    # abajur = base (cilindro) + haste (cilindro fino) + cupula (cone) + bulbo (esfera)
    ab_parts = []
    bv, bb = cilindro(raio=0.6, altura=0.25, cy=0.0, material="abj_base")
    ab_parts.append((bv, bb, {"abj_base": (0.20, 0.20, 0.22, 1.0)}))
    hv, hb = cilindro(raio=0.12, altura=1.3, cy=0.25, material="abj_haste")
    ab_parts.append((hv, hb, {"abj_haste": (0.30, 0.30, 0.32, 1.0)}))
    cv, cb = cone(raio=0.9, altura=0.9, cy=1.45, material="abj_cupula")
    ab_parts.append((cv, cb, {"abj_cupula": (0.95, 0.90, 0.55, 1.0)}))
    ev, eb = esfera_simples(raio=0.22, material="abj_bulbo")
    # sobe o bulbo para dentro da cupula
    ev = ev.reshape(-1, 8)
    ev[:, 1] += 1.55
    ev = ev.reshape(-1).astype(np.float32)
    ab_parts.append((ev, eb, {"abj_bulbo": (1.0, 0.95, 0.6, 1.0)}))

    # junta tudo num unico buffer reindexando os batches
    todos_v = []
    todos_b = []
    todas_cores = {}
    offset = 0
    for v, bts, cor in ab_parts:
        todos_v.append(v)
        for b in bts:
            todos_b.append({"material": b["material"], "start": offset + b["start"], "count": b["count"]})
        todas_cores.update(cor)
        offset += len(v) // 8
    abajur_v = np.concatenate(todos_v).astype(np.float32)
    _, abajur_b, abajur_m = montar_primitiva(abajur_v, todos_b, todas_cores)
    abajur_vao, abajur_vbo = criar_vao_vbo(abajur_v, loc_pos, loc_uv, loc_norm)
    abajur = {"vao": abajur_vao, "vbo": abajur_vbo, "batches": abajur_b, "materiais": abajur_m}

    # mesa de primitivas (interna)
    mesa_v, mesa_b = criar_mesa()
    _, mesa_b, mesa_m = montar_primitiva(mesa_v, mesa_b, {"mesa": (0.45, 0.30, 0.18, 1.0)})
    mesa_vao, mesa_vbo = criar_vao_vbo(mesa_v, loc_pos, loc_uv, loc_norm)
    mesa = {"vao": mesa_vao, "vbo": mesa_vbo, "batches": mesa_b, "materiais": mesa_m}

    # cadeira de primitivas (interna) - reutilizada em duas posicoes
    cad_v, cad_b = criar_cadeira()
    _, cad_b, cad_m = montar_primitiva(cad_v, cad_b, {"cadeira": (0.35, 0.22, 0.14, 1.0)})
    cad_vao, cad_vbo = criar_vao_vbo(cad_v, loc_pos, loc_uv, loc_norm)
    cadeira = {"vao": cad_vao, "vbo": cad_vbo, "batches": cad_b, "materiais": cad_m}

    # pequeno cubo emissivo p/ marcar o farol da bike e marcadores de luz
    marc_v, marc_b = primitiva(cubo(0, 0, 0, 1, 1, 1), "marcador")
    _, marc_b, marc_m = montar_primitiva(marc_v, marc_b, {"marcador": (1.0, 1.0, 0.85, 1.0)})
    marc_vao, marc_vbo = criar_vao_vbo(marc_v, loc_pos, loc_uv, loc_norm)
    marcador = {"vao": marc_vao, "vbo": marc_vbo, "batches": marc_b, "materiais": marc_m}

    # --- skybox -------------------------------------------------------------
    skybox_verts = skybox_vertices()
    skybox_vao, skybox_vbo = criar_vao_vbo(skybox_verts, loc_pos, loc_uv, loc_norm)
    skybox_tex = carregar_textura(skybox_tex_path, clamp=True) if os.path.exists(skybox_tex_path) else 0

    # --- estado de camera ---------------------------------------------------
    estado["bounds"] = bounds
    centro = (bounds[0] + bounds[1]) * 0.5
    estado["cam_pos"] = estado["bike_start_pos"].copy()
    dir_ini = centro - estado["cam_pos"]
    dir_ini_norm = dir_ini / np.linalg.norm(dir_ini)
    estado["cam_yaw"] = math.atan2(dir_ini_norm[0], dir_ini_norm[2])
    estado["cam_pitch"] = math.asin(float(np.clip(dir_ini_norm[1], -1.0, 1.0)))

    glEnable(GL_DEPTH_TEST)
    glClearColor(0.65, 0.75, 0.90, 1.0)

    # yaw fixo da bike (mesma logica do Projeto 2)
    bike_dir_v = estado["bike_end_pos"] - estado["bike_start_pos"]
    if np.linalg.norm(bike_dir_v) > 0.0:
        bike_yaw = math.atan2(bike_dir_v[0], bike_dir_v[2]) + estado["bike_yaw_offset"]
    else:
        bike_yaw = estado["bike_yaw_offset"]

    # posicoes internas fixas (relativas a casa)
    casa_pos = estado["casa_pos"]
    casa_alt = estado["casa_alt"]
    # lanterna pendurada no centro do teto
    lanterna_pos = casa_pos + np.array([0.0, casa_alt - 1.2, 0.0], dtype=np.float32)
    # a chama da lanterna fica um pouco acima da base do modelo
    luz_lanterna_pos = lanterna_pos + np.array([0.0, 0.4, 0.0], dtype=np.float32)
    # mesa centralizada num canto interno (origem da mesa fica no chao)
    mesa_pos = casa_pos + np.array([4.0, 0.0, -2.0], dtype=np.float32)
    # mesa: tampo 3.0 (X) x 2.0 (Z). As cadeiras vao em lados opostos no eixo Z,
    # ambas com o assento voltado para a mesa.
    # cadeira "da frente" (lado +Z da mesa): encosto deve ficar em +Z -> yaw = pi
    cadeira1_pos = mesa_pos + np.array([0.0, 0.0, 1.6], dtype=np.float32)
    cadeira1_yaw = math.pi
    # cadeira "de tras" (lado -Z da mesa): encosto em -Z -> yaw = 0
    cadeira2_pos = mesa_pos + np.array([0.0, 0.0, -1.6], dtype=np.float32)
    cadeira2_yaw = 0.0
    # abajur fica em cima do tampo da mesa (tampo ~2.6 de altura)
    abajur_pos = mesa_pos + np.array([0.0, 2.6, 0.0], dtype=np.float32)
    # a luz do abajur sai do bulbo (dentro da cupula)
    luz_abajur_pos = abajur_pos + np.array([0.0, 1.55, 0.0], dtype=np.float32)

    extras = [o for o in (bike, lanterna, mesa, cadeira) if o]

    tempo_ant = glfw.get_time()
    while not glfw.window_should_close(janela):
        agora = glfw.get_time()
        dt = agora - tempo_ant
        tempo_ant = agora

        glfw.poll_events()
        aplicar_controles(estado, dt)

        larg, alt = glfw.get_framebuffer_size(janela)
        asp = larg / max(alt, 1)
        proj = perspectiva(math.radians(60.0), asp, 0.1, 300.0)

        yaw, pitch = estado["cam_yaw"], estado["cam_pitch"]
        cam_dir = np.array([
            math.cos(pitch) * math.cos(yaw),
            math.sin(pitch),
            math.cos(pitch) * math.sin(yaw),
        ], dtype=np.float32)
        visao = look_at(estado["cam_pos"], estado["cam_pos"] + cam_dir)

        glViewport(0, 0, larg, alt)
        glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
        glUniformMatrix4fv(locs["proj"], 1, GL_TRUE, proj)
        glUniform3f(locs["cam"], *estado["cam_pos"])
        glUniform1f(locs["amb"], estado["luz_ambiente"])

        # ---- posicao atual da bike (e da sua luz/farol) -------------------
        t = estado["bike_t"]
        bike_pos = estado["bike_start_pos"] * (1.0 - t) + estado["bike_end_pos"] * t
        # farol um pouco a frente/acima do centro da bike
        frente = np.array([math.sin(bike_yaw - estado["bike_yaw_offset"]), 0,
                           math.cos(bike_yaw - estado["bike_yaw_offset"])], dtype=np.float32)
        luz_bike_pos = bike_pos + np.array([0.0, 3.2, 0.0], dtype=np.float32) + frente * 1.2

        # ---- envia as 3 fontes de luz -------------------------------------
        # indice 0: bike (EXTERNA, branca-amarelada)
        # indice 1: lanterna (INTERNA, quente)
        # indice 2: abajur (INTERNA, azulada)
        luz_pos = np.array([
            luz_bike_pos,
            luz_lanterna_pos,
            luz_abajur_pos,
        ], dtype=np.float32)
        luz_cor = np.array([
            [1.0, 0.95, 0.8],   # farol
            [1.0, 0.75, 0.4],   # lanterna quente
            [0.5, 0.65, 1.0],   # abajur azulado
        ], dtype=np.float32)
        luz_on = np.array([
            1.0 if estado["luz_bike_on"] else 0.0,
            1.0 if estado["luz_lanterna_on"] else 0.0,
            1.0 if estado["luz_abajur_on"] else 0.0,
        ], dtype=np.float32)
        luz_amb_id = np.array([0, 1, 1], dtype=np.int32)  # bike externa, resto interna

        glUniform1i(locs["nluz"], 3)
        glUniform3fv(loc_luz_pos, 3, luz_pos.reshape(-1))
        glUniform3fv(loc_luz_cor, 3, luz_cor.reshape(-1))
        glUniform1fv(loc_luz_on, 3, luz_on)
        glUniform1iv(loc_luz_amb, 3, luz_amb_id)

        kdm = estado["kd_mult"]
        ksm = estado["ks_mult"]

        # ---- skybox (emissiva, sem iluminacao) ----------------------------
        if skybox_tex:
            glDepthMask(GL_FALSE)
            glDisable(GL_DEPTH_TEST)
            view_sky = visao.copy()
            view_sky[0, 3] = view_sky[1, 3] = view_sky[2, 3] = 0.0
            glUniformMatrix4fv(locs["view"], 1, GL_TRUE, view_sky)
            glActiveTexture(GL_TEXTURE0)
            glBindTexture(GL_TEXTURE_2D, skybox_tex)
            desenhar_objeto({"vao": skybox_vao, "batches": [{"material":"sky","start":0,"count":len(skybox_verts)//8}],
                             "materiais": {"sky": {"cor": (1,1,1,1), "tex": skybox_tex}}},
                            locs, esc(250.0, 250.0, 250.0), 0, 0.0, 0.0, 1.0, emissivo=1)
            glBindTexture(GL_TEXTURE_2D, 0)
            glDepthMask(GL_TRUE)
            glEnable(GL_DEPTH_TEST)

        glUniformMatrix4fv(locs["view"], 1, GL_TRUE, visao)

        # ============ AMBIENTE EXTERNO (ambiente = 0) =======================
        # terreno: bastante difuso, pouco especular
        desenhar_objeto(outside, locs, ident(), 1, 0.9*kdm, 0.25*ksm, 8.0)

        # bike (externa) com seus proprios parametros
        if bike:
            bscale = estado["bike_scale"]
            tilt = estado["bike_tilt"]
            mat_bike = (transl(*bike_pos) @ rot_y(bike_yaw) @ rot_x(tilt)
                        @ esc(bscale, bscale, bscale))
            desenhar_objeto(bike, locs, mat_bike, 1, 0.8*kdm, 0.9*ksm, 24.0)

        # farol da bike: cubinho emissivo na posicao da luz externa (so se ligada)
        if estado["luz_bike_on"]:
            mat_farol = transl(*luz_bike_pos) @ esc(0.22, 0.22, 0.22)
            desenhar_objeto(marcador, locs, mat_farol, 0, 0, 0, 1, emissivo=1)

        # ============ AMBIENTE INTERNO (ambiente = 1) =======================
        # casa
        mat_casa = transl(*casa_pos)
        centro_casa = casa_pos + np.array([0.0, estado["casa_alt"]/2.0, 0.0], dtype=np.float32)
        desenhar_objeto(casa, locs, mat_casa, 3, 0.85*kdm, 0.5*ksm, 12.0,
                        split_casa=1, centro_obj=tuple(float(x) for x in centro_casa))

        # mesa e duas cadeiras (primitivas, internas)
        desenhar_objeto(mesa, locs, transl(*mesa_pos), 2, 0.8*kdm, 0.7*ksm, 16.0)
        desenhar_objeto(cadeira, locs,
                        transl(*cadeira1_pos) @ rot_y(cadeira1_yaw),
                        2, 0.8*kdm, 0.7*ksm, 16.0)
        desenhar_objeto(cadeira, locs,
                        transl(*cadeira2_pos) @ rot_y(cadeira2_yaw),
                        2, 0.8*kdm, 0.7*ksm, 16.0)

        # abajur sobre a mesa
        desenhar_objeto(abajur, locs, transl(*abajur_pos), 2, 0.8*kdm, 0.9*ksm, 20.0)
        # bulbo emissivo do abajur quando ligado
        if estado["luz_abajur_on"]:
            mat_bulbo = transl(*luz_abajur_pos) @ esc(0.2, 0.2, 0.2)
            desenhar_objeto(marcador, locs, mat_bulbo, 1, 0, 0, 1, emissivo=1)

        # lanterna pendurada no teto (interna)
        if lanterna:
            s = estado["lanterna_scale"] * estado["lanterna_base_scale"]
            mat_lant = transl(*lanterna_pos) @ esc(s, s, s)
            desenhar_objeto(lanterna, locs, mat_lant, 2, 0.8*kdm, 0.8*ksm, 18.0)
        # marcador emissivo da lanterna acesa
        if estado["luz_lanterna_on"]:
            mat_ml = transl(lanterna_pos[0], lanterna_pos[1] + 0.4, lanterna_pos[2]) @ esc(0.15, 0.15, 0.15)
            desenhar_objeto(marcador, locs, mat_ml, 1, 0, 0, 1, emissivo=1)

        glfw.swap_buffers(janela)

    # --- cleanup -----------------------------------------------------------
    todos_objetos = [outside, casa, abajur, marcador] + extras
    for obj in todos_objetos:
        glDeleteBuffers(1, [obj["vbo"]])
        glDeleteVertexArrays(1, [obj["vao"]])
    glDeleteBuffers(1, [skybox_vbo])
    glDeleteVertexArrays(1, [skybox_vao])
    if skybox_tex:
        glDeleteTextures(1, [skybox_tex])
    glDeleteProgram(prog)
    for obj in todos_objetos:
        for mat in obj["materiais"].values():
            if mat.get("tex"):
                glDeleteTextures(1, [mat["tex"]])
    glfw.terminate()

main()


## Resumo dos controles

**Camera**

| Tecla | Acao |
|-------|------|
| W / S | Frente / tras |
| A / D | Esquerda / direita |
| Q / E | Baixo / cima |
| Setas | Girar (yaw / pitch) |

**Iluminacao (requisitos do Projeto 3)**

| Tecla | Acao |
|-------|------|
| 1 | Liga/desliga luz da **bicicleta** (externa) |
| 2 | Liga/desliga **lanterna** do teto (interna) |
| 3 | Liga/desliga **abajur** da mesa (interno) |
| K / L | Diminui / aumenta **luz ambiente** |
| N / M | Diminui / aumenta **reflexao difusa** |
| , / . | Diminui / aumenta **reflexao especular** |
| 0 | Liga/desliga translacao da bicicleta |

A luz da bicicleta so afeta objetos do **ambiente externo**; lanterna e abajur so
afetam objetos do **ambiente interno**. Cada objeto tem seus proprios coeficientes
difuso/especular definidos no codigo (nenhum parametro de iluminacao vem de `.mtl`).
